# Hough lines demo

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets, IntSlider

In [ ]:
# generate some points
np.random.seed(10)
Width = 600
Height = 400

Npoints = 100
Npoint_rand =100

data_o = np.hstack( ( (np.random.rand(Npoints,1)-0.5)*200, (np.random.randn(Npoints,1)-0.5)*3 ) )
data_noise = np.random.rand(Npoint_rand,2)*np.array([Width,Height])

In [ ]:
def Hough( npt, nnoise, line_angle ):

  R = np.array( [ [np.cos(line_angle), -np.sin(line_angle)], [np.sin(line_angle), np.cos(line_angle)] ] )
  data = np.concatenate( (data_o[:npt,:]@R.T + np.array( [300.0,200.0]), data_noise[:nnoise,:]), axis=0 )

  r_max = np.sqrt( Width**2 + Height**2 )

  N_a = 180
  N_r = 100


  H = np.zeros( (N_r,N_a), dtype=int )

  for i in range(data.shape[0]):
    # for each point

    theta_i = np.arange( 0, N_a )
    theta = theta_i.astype(float) / N_a * np.pi
    r = data[i,0] * np.cos(theta) + data[i,1] * np.sin(theta)
    r_i = np.round( (r+r_max)/(2*r_max) * N_r).astype(int)
    H[ r_i , theta_i ] += 1


  max_r_i, max_theta_i = np.unravel_index( np.argmax( H ), (N_r,N_a) )

  # Plotting

  max_theta = max_theta_i.astype(float) / N_a * np.pi
  max_r = 2 * r_max * float(max_r_i) / N_r - r_max

  c = np.array( [max_r*np.cos(max_theta), max_r*np.sin(max_theta)] )
  p1 = c + np.array([-np.sin(max_theta), np.cos(max_theta)]) * 1000
  p2 = c - np.array([-np.sin(max_theta), np.cos(max_theta)]) * 1000


  fig, (ax1, ax2) = plt.subplots( 1,2, figsize=(14,7), width_ratios=[1.5,1] )

  ax1.scatter( data[:,0], data[:,1], s=30, c='k')
  if npt > 1:
    ax1.plot( [p1[0],p2[0]], [p1[1],p2[1]], '-.r', linewidth=1 )
  ax1.set_xlim(0,Width)
  ax1.set_ylim(0,Height)
  ax1.set_aspect('equal')
  ax1.set_title('Points')
  ax1.set_xlabel('x')
  ax1.set_ylabel('y')
  ax1.grid(True)

  XX,YY = np.meshgrid(np.linspace(0,np.pi,H.shape[1]), np.linspace(-r_max,r_max,H.shape[0])  )
  ax2.pcolor(XX,
             YY,
             H, cmap="gray_r" )
  ax2.set_xlabel('theta')
  ax2.set_ylabel('r')
  ax2.set_title('Hough accumulator')
  ax2.grid(True)


interact( Hough,
          npt=IntSlider(description='n. inliers',min=0,max=Npoints,step=1,value=0),
          nnoise=IntSlider(description='n. outliers',min=0,max=Npoint_rand,step=1,value=0),
          line_angle=widgets.FloatSlider(description='Line rotation', min=0,max=np.pi,step=0.1,value=150 / 180.0 * np.pi) )